# 🚀 Clasificador de Frutas TP — Etapa 2: Transfer Learning

## UTN-IA 2026 · Food Classification - Training Set

**Competencia:** https://www.kaggle.com/competitions/utn-ia-2026-fruit-vegetable-classifier

---

### ¿Por qué un notebook nuevo?

El notebook original (`Clasificador_Frutas_UTN.ipynb`) es el **baseline de la cursada**. Acá vamos a iterar para subir el puntaje de la competencia.

### ¿Por qué arrancamos desde cero con el split?

Porque el split del notebook original tenía un **data leak**: TODAS las imágenes de `val` y `test` estaban también en `train` (overlap 100%). El 71.7% de val_acc era **mentira**: el modelo evaluaba en imágenes que ya había visto.

En este notebook lo arreglamos para que el split sea **limpio y verificado** — si hay overlap, la celda **falla antes de entrenar**.

### Plan (mismo orden que el Trabajo Práctico del notebook original)

1. **Transfer Learning** — ResNet18 preentrenado en ImageNet, backbone congelado, clasificador nuevo
2. **Fine-tuning gradual** — descongelar capas superiores con LR diferencial
3. **Augmentation agresivo** — MixUp, CutMix, RandAugment
4. **Ensemble** — promediar varios modelos
5. **Test-Time Augmentation (TTA)** — predecir con múltiples crops y promediar


## 0. Setup

In [ ]:
import os
import random
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Reproducibilidad
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'PyTorch {torch.__version__} | Torchvision {torchvision.__version__}')

def worker_init_fn(worker_id):
    '''Fuerza a cada worker del DataLoader a importar numpy (workaround bug fork+numpy 2.x).'''
    import numpy

print('--- Imports OK ---')


## 1. Dataset y split LIMPIO (sin data leak)

El dataset crudo `notebook_frutas_recognition/train/` tiene TODAS las imágenes en una sola carpeta por clase (sin split previo).
Vamos a hacer el split nosotros, **verificando que no haya overlap** entre train/val/test.

> Si después del split aparece overlap (mismo filename en train y val), la celda FALLA. Es un guardián de seguridad — preferimos que reviente acá y no después de entrenar.


In [ ]:
# Dataset crudo (todo en train/clase/)
RAIZ_Crudo = Path('notebook_frutas_recognition/train')
DESTINO_SPLIT = Path('notebook_frutas_tp_split')

CLASES = ['apple', 'banana', 'grapes', 'potato', 'tomato']
NUM_CLASES = len(CLASES)

SPLIT_TRAIN, SPLIT_VAL, SPLIT_TEST = 0.70, 0.15, 0.15

def split_limpio(raiz, destino, clases, pct_train=0.70, pct_val=0.15, seed=42):
    rng = random.Random(seed)
    if destino.exists():
        shutil.rmtree(destino)
    destino.mkdir(parents=True)

    conteo = {}
    for clase in clases:
        carpeta_clase = raiz / clase
        if not carpeta_clase.exists():
            raise FileNotFoundError(f'No existe {carpeta_clase}')
        imgs = sorted([
            p for p in carpeta_clase.iterdir()
            if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
        ])
        rng.shuffle(imgs)
        n = len(imgs)
        n_val = int(n * pct_val)
        n_test = int(n * (1 - pct_train - pct_val))
        # Splits disjuntos por slicing (no se pisan)
        splits = {
            'val':   imgs[:n_val],
            'test':  imgs[n_val:n_val + n_test],
            'train': imgs[n_val + n_test:],
        }
        for split_name, subset in splits.items():
            dst_dir = destino / split_name / clase
            dst_dir.mkdir(parents=True, exist_ok=True)
            for img in subset:
                shutil.copy(img, dst_dir / img.name)
        conteo[clase] = {'train': len(splits['train']), 'val': len(splits['val']), 'test': len(splits['test'])}
    return conteo

conteo = split_limpio(RAIZ_Crudo, DESTINO_SPLIT, CLASES)

print(f'Split creado en {DESTINO_SPLIT}/')
for c in CLASES:
    cc = conteo[c]
    print(f'  {c:10s}  train={cc["train"]:3d}  val={cc["val"]:3d}  test={cc["test"]:3d}')


In [ ]:
def verificar_no_leak(destino, clases):
    '''Falla si algun (clase,filename) aparece en 2 splits distintos.'''
    splits = ['train', 'val', 'test']
    por_split = {}
    for split in splits:
        pares = set()
        for c in clases:
            carpeta = destino / split / c
            if carpeta.exists():
                for p in carpeta.iterdir():
                    if p.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                        pares.add(f'{c}/{p.name}')
        por_split[split] = pares
    overlap = []
    for i, s1 in enumerate(splits):
        for s2 in splits[i+1:]:
            inter = por_split[s1] & por_split[s2]
            if inter:
                overlap.append((s1, s2, inter))
    if overlap:
        for s1, s2, inter in overlap:
            print(f'❌ LEAK entre {s1} y {s2}: {len(inter)} archivos compartidos')
            for x in list(inter)[:5]:
                print(f'   {x}')
        raise RuntimeError(f'Data leak detectado: {len(overlap)} par(es) con overlap. No entrenar asi.')
    print('✅ Sin data leak: train, val y test son disjuntos.')

verificar_no_leak(DESTINO_SPLIT, CLASES)

total = sum(conteo[c][s] for c in CLASES for s in ['train','val','test'])
print(f'Total imagenes: {total}')


## 2. Transformaciones (224x224)

**¿Por qué 224x224 y no 64x64?** Porque ResNet18 fue preentrenada en ImageNet con esa resolución. Sus filtros aprendieron a detectar patrones a esa escala. Si le tiramos 64x64, destruimos el conocimiento preentrenado (menos detalles que los filtros esperan) y perdemos la ventaja del Transfer Learning.

Augmentation solo en train. Val y test siempre el mismo pipeline base para comparar justamente.


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

# ImageNet stats (lo que vieron los modelos preentrenados)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Pipeline base: val y test
transform_base = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# Pipeline con augmentation: SOLO training
# Nota: SACAMOS RandomVerticalFlip porque una manzana al reves no es mas manzana
# (aprende algo que despues no aparece en test). Flip horizontal SI: un objeto real
# puede verse al espejo sin perder sentido. Rotacion 15 grados SI. Color jitter SI.
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

print(f'IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}')
print('Train: resize+crop+flip_H+rotacion+colorjitter+normalize')
print('Val/Test: resize+normalize (sin augmentation)')


## 3. DataLoaders

Algunas imágenes del dataset estaban con formato transparente (RGBA o P). Usamos un `loader` custom que las convierte a RGB sobre fondo blanco, igual que el notebook original.


In [ ]:
def cargar_imagen_limpia(path):
    '''Convierte formatos problematicos (RGBA/P) a RGB sobre fondo blanco.'''
    img = Image.open(path)
    if img.mode in ('P', 'RGBA', 'LA'):
        img = img.convert('RGBA')
        fondo = Image.new('RGB', img.size, (255, 255, 255))
        fondo.paste(img, mask=img.split()[3])
        return fondo
    return img.convert('RGB')

BASE_DIR = DESTINO_SPLIT

train_ds = ImageFolder(str(BASE_DIR / 'train'), transform=transform_train, loader=cargar_imagen_limpia)
val_ds   = ImageFolder(str(BASE_DIR / 'val'),   transform=transform_base,   loader=cargar_imagen_limpia)
test_ds  = ImageFolder(str(BASE_DIR / 'test'),  transform=transform_base,   loader=cargar_imagen_limpia)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True, worker_init_fn=worker_init_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, worker_init_fn=worker_init_fn)
test_loader  = DataLoader(test_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, worker_init_fn=worker_init_fn)

class_names = train_ds.classes
print(f'Datasets: train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')
print(f'Clases (ImageFolder las ordena alfabeticamente): {class_names}')

imgs, labels = next(iter(train_loader))
print(f'Forma del batch: {imgs.shape}  → [batch, canales, H, W]')


## 4. Vistazo rápido

In [ ]:
def mostrar_ejemplos(ds, n=3):
    fig, axes = plt.subplots(len(CLASES), n, figsize=(n*3, len(CLASES)*3))
    for i, clase in enumerate(CLASES):
        idx_clase = class_names.index(clase)
        idxs = [j for j, (_, l) in enumerate(ds) if l == idx_clase][:n]
        for j, idx in enumerate(idxs):
            img, _ = ds[idx]
            # Desnormalizar para ver
            img = img * torch.tensor(STD).view(3,1,1) + torch.tensor(MEAN).view(3,1,1)
            axes[i][j].imshow(img.permute(1,2,0).clamp(0,1).numpy())
            axes[i][j].axis('off')
            if j == 0:
                axes[i][j].set_ylabel(clase, fontsize=11, fontweight='bold')
    plt.suptitle('Dataset Etapa 2 (test, sin augmentation)', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

# Usamos val_ds que tiene transform_base (sin augmentation) para ver las imagenes reales
mostrar_ejemplos(val_ds, n=3)


## 5. Transfer Learning con ResNet18

Receta:
1. Cargar ResNet18 con pesos `IMAGENET1K_V1` (entrenado en 1.2M imágenes, 1000 clases)
2. **Congelar el backbone**: `requires_grad=False` en TODOS los parámetros de las capas convolucionales
3. **Reemplazar la cabeza clasificadora**: la última capa de ResNet18 es `Linear(512, 1000)`. La sacamos y ponemos un clasificador nuevo para 5 clases.

Solo el clasificador nuevo aprende (~131K parámetros). El backbone ya sabe ver.


In [ ]:
modelo_tl = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Congelamos el backbone: requires_grad=False en todos los params convolucionales
for param in modelo_tl.parameters():
    param.requires_grad = False

# Reemplazamos la cabeza (ultima capa fc) por un clasificador nuevo para 5 clases
num_features = modelo_tl.fc.in_features  # 512 en ResNet18
modelo_tl.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(256, NUM_CLASES),
)

modelo_tl = modelo_tl.to(device)

# Solo contamos los params entrenables (los del clasificador nuevo)
trainable = sum(p.numel() for p in modelo_tl.parameters() if p.requires_grad)
total = sum(p.numel() for p in modelo_tl.parameters())
print(f'Modelo: ResNet18 + clasificador nuevo')
print(f'  Total parametros: {total:,}')
print(f'  Entrenables: {trainable:,}  (clasificador nuevo)')
print(f'  Congelados: {total - trainable:,}  (backbone preentrenado)')


In [ ]:
with torch.no_grad():
    x = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(device)
    y = modelo_tl(x)
    print(f'Test de dimensiones:')
    print(f'  Entrada: {x.shape}')
    print(f'  Salida:  {y.shape}  → 1 score por clase')


## 6. Entrenamiento (solo el clasificador nuevo)

Como el backbone está congelado, solo se actualizan los pesos del clasificador nuevo. Es rápido y converge con pocos epochs.


In [ ]:
EPOCHS = 10
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    [p for p in modelo_tl.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-4,
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

def entrenar_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correcto, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        _, preds = out.max(1)
        total_loss += loss.item() * imgs.size(0)
        correcto   += preds.eq(labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correcto / total

@torch.no_grad()
def evaluar(model, loader, criterion, device):
    model.eval()
    total_loss, correcto, total = 0, 0, 0
    preds_all, labels_all = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        loss = criterion(out, labels)
        _, preds = out.max(1)
        total_loss += loss.item() * imgs.size(0)
        correcto   += preds.eq(labels).sum().item()
        total      += imgs.size(0)
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    f1w = f1_score(labels_all, preds_all, average='weighted')
    return total_loss / total, correcto / total, f1w

historial = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'val_f1w':[]}
mejor_val_f1w = 0.0
CKPT_PATH = 'notebook_frutas_tp_resnet18.pth'

print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>10} | {"Val Loss":>10} | {"Val Acc":>10} | {"Val F1w":>8}')
print('-' * 70)

for epoch in range(1, EPOCHS + 1):
    tl, ta = entrenar_epoch(modelo_tl, train_loader, criterion, optimizer, device)
    vl, va, vf1 = evaluar(modelo_tl, val_loader, criterion, device)
    scheduler.step(vl)
    for k, v in [('train_loss',tl),('val_loss',vl),('train_acc',ta),('val_acc',va),('val_f1w',vf1)]:
        historial[k].append(v)
    marca = ''
    if vf1 > mejor_val_f1w:
        mejor_val_f1w = vf1
        torch.save({
            'epoch': epoch,
            'model_state_dict': modelo_tl.state_dict(),
            'val_acc': va,
            'val_f1w': vf1,
            'class_names': class_names,
        }, CKPT_PATH)
        marca = ' ✅'
    print(f'{epoch:>6} | {tl:>10.4f} | {ta:>9.1%} | {vl:>10.4f} | {va:>9.1%} | {vf1:>8.4f}{marca}')

print(f'\nMejor Val F1w: {mejor_val_f1w:.4f}')


In [ ]:
def graficar_historial(hist):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    ep = range(1, len(hist['train_loss'])+1)

    axes[0].plot(ep, hist['train_loss'], 'b-o', ms=3, label='Train')
    axes[0].plot(ep, hist['val_loss'],   'r-o', ms=3, label='Val')
    axes[0].set_title('Pérdida'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(ep, [a*100 for a in hist['train_acc']], 'b-o', ms=3, label='Train')
    axes[1].plot(ep, [a*100 for a in hist['val_acc']],   'r-o', ms=3, label='Val')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylim([0,105]); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(ep, hist['val_f1w'], 'g-o', ms=4, label='Val F1w')
    best_ep = int(np.argmax(hist['val_f1w']))+1
    axes[2].axvline(x=best_ep, color='gray', ls='--', alpha=0.7, label=f'Mejor epoch {best_ep}')
    axes[2].set_title('Val F1 Weighted (m\u00e9trica de la competencia)'); axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.suptitle('Transfer Learning (ResNet18, backbone congelado) -- clase', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

graficar_historial(historial)


In [ ]:
# Cargar mejor modelo
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
modelo_tl.load_state_dict(ckpt['model_state_dict'])
print(f'Mejor modelo cargado: epoch {ckpt["epoch"]}  val_acc={ckpt["val_acc"]:.2%}  val_f1w={ckpt["val_f1w"]:.4f}')

# Evaluar en test
test_loss, test_acc, test_f1w = evaluar(modelo_tl, test_loader, criterion, device)
print(f'Test: loss={test_loss:.4f}  acc={test_acc:.2%}  F1w={test_f1w:.4f}')

# Predicciones + classification report
@torch.no_grad()
def obtener_preds(model, loader, device):
    model.eval()
    preds_all, labels_all = [], []
    for imgs, labels in loader:
        out = model(imgs.to(device))
        _, preds = out.max(1)
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(labels.numpy())
    return np.array(preds_all), np.array(labels_all)

preds, labels_r = obtener_preds(modelo_tl, test_loader, device)
print()
print(classification_report(labels_r, preds, target_names=class_names, digits=4))


## 7. Comparación con el baseline de la cursada

El notebook original alcanzó **71.7% val_acc** con `FrutasCNN` entrenada from scratch en un dataset con data leak.

**IMPORTANTE:** ese 71.7% no es comparable con nuestro `val_f1w` acá, porque:
- Distinto dataset de val (el original estaba contaminado)
- Distinta métrica (accuracy vs F1 weighted)

La comparación justa es en la **competencia**. Por eso el siguiente paso es generar submission y mandarla.


## 8. Submission a la competencia

Generamos `submission.csv` con el modelo de Transfer Learning y lo dejamos listo para subir.

**Formato Kaggle:**
```image_id,label
img_00000,apple
img_00001,tomato
...
```


In [ ]:
import pandas as pd

CLASES_COMP = CLASES  # ['apple', 'banana', 'grapes', 'potato', 'tomato'] (orden alfabético = orden ImageFolder)

DIR_TEST = Path('data_competencia')
imagenes_test = sorted([
    p for p in DIR_TEST.rglob('*')
    if p.suffix.lower() in ['.jpg', '.jpeg', '.png']
])
print(f'{len(imagenes_test)} imagenes de test de la competencia')

modelo_submit = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
modelo_submit.fc = nn.Sequential(
    nn.Linear(512, 256), nn.ReLU(), nn.Dropout(p=0.4), nn.Linear(256, NUM_CLASES)
)
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
modelo_submit.load_state_dict(ckpt['model_state_dict'])
modelo_submit = modelo_submit.to(device)
modelo_submit.eval()
print(f'Checkpoint cargado: epoch {ckpt["epoch"]}  val_f1w={ckpt["val_f1w"]:.4f}')

predicciones = []
with torch.no_grad():
    for img_path in imagenes_test:
        img = cargar_imagen_limpia(img_path)
        tensor = transform_base(img).unsqueeze(0).to(device)
        logits = modelo_submit(tensor)
        pred_idx = logits.argmax(dim=1).item()
        pred_label = CLASES_COMP[pred_idx]
        predicciones.append({
            'image_id': os.path.splitext(img_path.name)[0],
            'label': pred_label,
        })

df_sub = pd.DataFrame(predicciones)
print(df_sub.head(10))
print(f'\nDistribucion de predicciones:')
print(df_sub['label'].value_counts())


In [ ]:
# Validar clases
clases_invalidas = df_sub[~df_sub['label'].isin(CLASES_COMP)]
if len(clases_invalidas) > 0:
    print(f'⚠️  {len(clases_invalidas)} predicciones con clases invalidas')
    print(clases_invalidas)
else:
    print(f'✅ Todas las {len(df_sub)} predicciones tienen clases validas')

# Guardar
df_sub.to_csv('submission_tl.csv', index=False)
print(f'📄 submission_tl.csv guardado')
print(f'  Subilo en: https://www.kaggle.com/competitions/utn-ia-2026-fruit-vegetable-classifier/submit')


## 9. Próximos pasos (para seguir sumando)

1. **Fine-tuning gradual** — descongelar las últimas capas del backbone (`layer4` de ResNet) y seguir entrenando con LR mucho más chico (e-5). Es lo que más sube el score.
2. **Augmentation más agresivo** — RandAugment, MixUp o CutMix.
3. **Ensemble** — entrenar 2-3 modelos (ResNet18, ResNet34, EfficientNet-B0) y promediar logits.
4. **Test-Time Augmentation (TTA)** — predecir con varios crops+flip y promediar. Sube ~1-2% sin reentrenar.
